# Модель B на сырых сделках BTC/ETH — запуск из браузера
Runtime → Change runtime type → **A100 GPU**, затем Runtime → Run all. Вкладку держать открытой.
Данные (~6 ГБ) и чекпоинты пишутся на Google Drive (`MyDrive/xauusd_seq`), поэтому если сессия оборвётся — просто снова Run all: обучение продолжится с последнего чекпоинта (каждые 1000 шагов).
Ожидаемо ~4 ч A100 (~21 юнит) с нуля.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/xauusd && git clone -q --depth 1 -b claude/diag-meta-gradient-budget https://github.com/Flyingzaptop/xauusd.git /content/xauusd
!ln -sfn /content/xauusd/experiments/seqmodel /content/seq
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
os.environ['DATA_DIR']='/content/drive/MyDrive/xauusd_seq/data'
os.environ['OUT_DIR']='/content/drive/MyDrive/xauusd_seq/outB'
# builds any missing monthly npz from data.binance.vision (first run ~10-15 min), then trains / resumes
!cd /content/seq && python -u vm_prep.py

In [ ]:
import json;r=json.load(open('/content/drive/MyDrive/xauusd_seq/outB/result.json'))
for k,v in r['final'].items():
    print(k);[print(f"  {kk}: ic {vv['ic']:+.4f} gross {vv['top5_gross_bp']:+.2f} net@4 {vv['net4']:+.2f} n {vv['n']}") for kk,vv in v.items()]